In [25]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
def prepare_data(input_file_path, sol):

    df = pd.read_csv(input_file_path, low_memory=False)
    columns_for_X = [f'{sol}_seq_A2', f'{sol}_seq_A3',f'{sol}_seq_A4',f'{sol}_seq_A5',f'{sol}_seq_A6',
                    f'{sol}_seq_C2', f'{sol}_seq_C3',f'{sol}_seq_C4',f'{sol}_seq_C5',f'{sol}_seq_C6',
                    f'{sol}_seq_G2', f'{sol}_seq_G3',f'{sol}_seq_G4',f'{sol}_seq_G5',f'{sol}_seq_G6',
                    f'{sol}_seq_T2', f'{sol}_seq_T3',f'{sol}_seq_T4',f'{sol}_seq_T5',f'{sol}_seq_T6']

    X = df[columns_for_X]
    X = X.dropna(axis=0, how='any')

    y = df[f'{sol}_FRET']

    new_feature_name = ['A1','A2','A3','A4','A5','C1','C2','C3','C4','C5','G1','G2','G3','G4','G5','T1','T2','T3','T4','T5']
    X.columns = new_feature_name

    return X, y


def prepare_data_3dots(input_file_path, sol):

    df = pd.read_csv(input_file_path, low_memory=False)
    columns_to_drop = ['N5_seq','N5_FRET','N50_seq','N50_FRET','N500_seq','N500_FRET','N5M10_seq','N5M10_FRET','N5M100_seq','N5M100_FRET']

    X = df.drop(columns=columns_to_drop)
    X = X.dropna(axis=0, how='any')
    X = X.astype(np.int32)
    
    y = df[f'{sol}_FRET']
    y = y.dropna()

    return X, y


def baseline_error(y_train, y_test):
    baseline_value = np.mean(y_train)
    baseline_predictions = np.full_like(y_test, baseline_value)
    
    baseline_mae = mean_absolute_error(y_test, baseline_predictions)
    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
    # baseline_mse = mean_squared_error(y_test, baseline_predictions)
    return baseline_mae, baseline_rmse #, baseline_mse


In [28]:
# Define the objective function for Hyperopt 
def objective(params):
    num_round = 100  # Number of boosting rounds
    
    # Train the model
    bst = xgb.train(params, dtrain, num_round)
    
    # Make predictions
    preds = bst.predict(dtest)
    
    # Calculate error
    # mse = mean_squared_error(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    
    # Correctly format the numeric values in the params dictionary
    formatted_params = {k: f"{v:.4f}" if isinstance(v, float) else v for k, v in params.items()}

    # Print the formatted parameters and MAE
    print(f"Trial with params: {formatted_params}, MAE: {mae:.4f}")
    
    # Hyperopt tries to minimize the objective function, so return the mse
    return {'loss': mae, 'status': STATUS_OK}


# Custom objective function for MAE
def mae_obj(preds, dtrain):
    labels = dtrain.get_label()
    grad = np.sign(preds - labels)  # Gradient
    hess = np.ones_like(grad)       # Hessian (second derivative)
    return grad, hess


In [ ]:
solution = ['N5', 'N50', 'N500', 'N5M10', 'N5M100']

# Define the hyperparameter space
space = {
    'max_depth': hp.choice('max_depth', range(5, 10)),
    'eta': hp.uniform('eta', 0.01, 0.3),
    'gamma': hp.uniform('gamma', 0, 1),
    'subsample': hp.uniform('subsample', 0.5, 1),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
    'objective': 'reg:squarederror',  # Objective function for regression
    'lambda': hp.uniform('lambda', 1e-3, 10),  # L2 regularization term on weights
    'alpha': hp.uniform('alpha', 1e-3, 10)    # L1 regularization term on weights
}


for sol in solution:
    
    input_file_path = f'INPUT_FILE_PATH'

    # prepare X, y
    X, y = prepare_data_3dots(input_file_path, sol)
    
    # Split into train and test dataset
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # DMatrix format for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)
    
    baseline_mae, baseline_rmse = baseline_error(y_train, y_test)

    print(f'{sol}\nbaseline_mae:{baseline_mae:.4f}\nbaseline_rmse:{baseline_rmse:.4f}\n')

    # Run the Hyperopt optimization
    trials = Trials()

    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=50,
        trials=trials
    )
    
    print(f"Best hyperparameters of {sol}:", best)

    # Calculate the best loss (MAE)
    best_loss = min(trials.results, key=lambda x: x['loss'])['loss']
    print(f"Best MAE of {sol}: {best_loss:.4f}\n")

In [ ]:
solution = ['N5', 'N50', 'N500', 'N5M10', 'N5M100']

# Define the hyperparameter space
space = {
    'max_depth': hp.choice('max_depth', range(5, 10)),
    'eta': hp.uniform('eta', 0.01, 0.3),
    'gamma': hp.uniform('gamma', 0, 1),
    'subsample': hp.uniform('subsample', 0.5, 1),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
    'objective': 'reg:squarederror',  # Objective function for regression
    'lambda': hp.uniform('lambda', 1e-3, 10),  # L2 regularization term on weights
    'alpha': hp.uniform('alpha', 1e-3, 10)    # L1 regularization term on weights
}


for sol in solution:
    
    input_file_path_3dots = f'nput_file_path_for_3dots_patterns'

    # prepare X, y
    X, y = prepare_data_3dots(input_file_path_3dots, sol)
    
    # Split into train and test dataset
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # DMatrix format for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)
    
    baseline_mae, baseline_rmse = baseline_error(y_train, y_test)

    print(f'{sol}\nbaseline_mae:{baseline_mae:.4f}\nbaseline_rmse:{baseline_rmse:.4f}\n')

    # Run the Hyperopt optimization
    trials = Trials()

    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=50,
        trials=trials
    )
    
    print(f"Best hyperparameters of {sol}:", best)

    # Calculate the best loss (MAE)
    best_loss = min(trials.results, key=lambda x: x['loss'])['loss']
    print(f"Best MAE of {sol}: {best_loss:.4f}\n")